In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
%matplotlib inline

In [2]:
df=pd.read_csv('F:\DataTalkClubs\ML_Zoomcamp\car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df=df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]

###  Missing values 

In [4]:
df.isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

### Median for horsepower

In [5]:
df['horsepower'].median()

np.float64(254.0)

### Prepare and split the dataset

In [6]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [7]:
y_train=df_train.fuel_efficiency_mpg.values
y_val=df_val.fuel_efficiency_mpg.values
y_test=df_test.fuel_efficiency_mpg.values

In [8]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

### Filling NAs 

In [9]:
columns = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]

In [10]:
def train_linear_regression(X,y):
    ones=np.ones(X.shape[0])
    X=np.column_stack([ones, X])

    XTX=X.T.dot(X)
    XTX_inv=np.linalg.inv(XTX)
    w=XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def linear_regression(X,w0,w):
    return w0 + X.dot(w)

In [11]:
def rmse(y,y_pred):
    se=(y-y_pred)**2
    mse=se.mean()
    return round(np.sqrt(mse), 4)
    

#### Fill with 0

In [12]:
def prepare_X(df):
    df=df[columns]
    return df.fillna(0).values    

In [13]:
X_train=prepare_X(df_train)
w0,w=train_linear_regression(X_train, y_train)

In [14]:
X_val=prepare_X(df_val)
y_pred=linear_regression(X_val, w0,w)
rmse_0=rmse(y_val, y_pred)

#### Fill with mean

In [15]:
mean=df_train.horsepower.mean()
mean

np.float64(254.46338337605272)

In [16]:
def prepare_X_mean(df, mean):
    df=df[columns]
    df['horsepower']=df['horsepower'].fillna(mean)
    return df.values

In [17]:
X_train=prepare_X_mean(df_train, mean)
w0, w=train_linear_regression(X_train, y_train)

In [18]:
X_val=prepare_X(df_val)
y_pred=linear_regression(X_val, w0, w)
rmse_mean=rmse(y_val, y_pred)

In [23]:
if rmse_0<rmse_mean:
    print('RMSE Zero:', rmse_0)
else:
    print('With mean:', rmse_mean)

print(rmse_0, rmse_mean)

RMSE Zero: 2.2053
2.2053 2.2323


### Best regularization

In [33]:
def train_linear_regression_reg(X,y,r):
    ones=np.ones(X.shape[0])
    X=np.column_stack([ones, X])

    XTX=X.T.dot(X)
    XTX=XTX+r*np.eye(XTX.shape[0])
    XTX_inv=np.linalg.inv(XTX)
    w=XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]
    

In [29]:
for r in [0,0.01,0.1,1, 5, 10, 100]:
    X_train=prepare_X(df_train)
    w0,w=train_linear_regression_reg(X_train, y_train, r)

    X_val=prepare_X(df_val)
    y_pred=linear_regression(X_val, w0, w)
    score=rmse(y_val, y_pred)
    print('R',r)
    print('Score',score)

R 0
Score 2.2053
R 0.01
Score 2.2058
R 0.1
Score 2.2241
R 1
Score 2.3492
R 5
Score 2.4094
R 10
Score 2.4195
R 100
Score 2.4292


### RMSE standard deviation

In [34]:
n=len(df)
print(n)

10000


In [35]:
n_val=int(n*0.2)
n_test=int(n*0.2)
n_train=n-n_val-n_test
print(n_train)

6000


In [52]:
scores = []

for s in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values

    del df_train["fuel_efficiency_mpg"]
    del df_val["fuel_efficiency_mpg"]
    del df_test["fuel_efficiency_mpg"]

    X_train = prepare_X(df_train)
    w0, w = train_linear_regression(X_train, y_train)

    X_val = prepare_X(df_val)
    y_pred = linear_regression(X_val, w0, w)
    scores.append(rmse(y_val, y_pred))

round(np.std(scores),3)

np.float64(0.029)

### Evaluation on test

In [53]:
n=len(df)
n_val=int(n*0.2)
n_test=int(n*0.2)
n_train=n-n_val-n_test
n_train

6000

In [54]:
np.random.seed(9)
idx=np.arange(n)
np.random.shuffle(idx)


In [55]:
df_train=df.iloc[idx[:n_train]]
df_val=df.iloc[idx[n_train:n_train+n_val]]
df_test=df.iloc[idx[n_train+n_val:]]

In [56]:
df_train.head(2)

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
3644,2320,246.0,4100,2014,32.5
9184,2230,252.0,4250,1998,33.2


In [58]:
y_train=df_train.fuel_efficiency_mpg.values
y_val=df_val.fuel_efficiency_mpg.values
y_test=df_test.fuel_efficiency_mpg.values

In [60]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [62]:
df_full=pd.concat([df_train, df_val], ignore_index=True)
X_full=prepare_X(df_full)
y_full=np.concat((y_train, y_val))
w0,w=train_linear_regression_reg(X_full, y_full, 0.001)

In [63]:
X_test=prepare_X(df_test)
y_pred=linear_regression(X_test, w0, w)
round(rmse(y_test, y_pred), 3)

np.float64(2.236)